# Test court de correction vers un modele pinhole

But : partir de `00000.jpg`, reconstruire `K` depuis `camera.txt`, puis tester une correction simple avec `d`.

Comme le dataset ne donne qu'un seul `d`, on teste les deux interpretations les plus simples : `+d` et `-d` comme coefficient radial `k1`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
image_path = r"C:\Users\conqu\Videos\Detection_Deep_Fake\Répo_1st_test\Implémentaiton_1\dataset\real\sequence_01\images\00000.jpg"

buffer = np.fromfile(image_path, dtype=np.uint8)
img_bgr = cv2.imdecode(buffer, cv2.IMREAD_COLOR)
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

h, w = img.shape[:2]
print(f"Resolution : {w} x {h}")

In [ ]:
# Parametres 1280x1024 lus dans camera.txt
fx_n = 0.349153
fy_n = 0.436593
cx_n = 0.493140
cy_n = 0.499021
d = 0.933271

K = np.array([
    [fx_n * w, 0, cx_n * w],
    [0, fy_n * h, cy_n * h],
    [0, 0, 1],
], dtype=np.float64)

print("K =")
print(K)
print("d =", d)

In [ ]:
def undistort_with_k1(image_rgb, K, k1, alpha=0.0):
    dist = np.array([k1, 0.0, 0.0, 0.0, 0.0], dtype=np.float64)
    new_K, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
    map1, map2 = cv2.initUndistortRectifyMap(K, dist, None, new_K, (w, h), cv2.CV_32FC1)
    corrected = cv2.remap(image_rgb, map1, map2, interpolation=cv2.INTER_LINEAR)
    return corrected, new_K

img_plus, K_plus = undistort_with_k1(img, K, d)
img_minus, K_minus = undistort_with_k1(img, K, -d)

print("K_plus =")
print(K_plus)
print()
print("K_minus =")
print(K_minus)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(img)
ax[0].set_title("Originale")
ax[0].axis("off")

ax[1].imshow(img_plus)
ax[1].set_title("Correction avec +d")
ax[1].axis("off")

ax[2].imshow(img_minus)
ax[2].set_title("Correction avec -d")
ax[2].axis("off")

plt.tight_layout()

Lecture rapide :

- si `+d` ou `-d` redresse mieux les structures droites, garde ce cas
- si les deux changent tres peu, alors soit la distorsion est faible, soit `d` ne suit pas le modele radial OpenCV